# **Day 56: Decorators (Part 1)**

A **decorator** is a function that **adds extra behaviour** to another function, **without changing its code**.

**Real-life example:** Think of **gift wrapping**. The gift inside stays the same, but the wrapping adds something extra around it.

We have already used decorators like `@property`, `@classmethod`, `@dataclass` and `@lru_cache`. Today we learn how they work and how to make our own.

## **Three Things We Need to Know First**
1. A function can be **stored in a variable** and **passed** to another function.
2. A function can be **written inside** another function.
3. A function can **return** another function.

In [1]:
def shout(text):
    return text.upper()

def run(func, value):          # A function that receives another function
    return func(value)

print(run(shout, "jai bangla"))

JAI BANGLA


## **Making a Decorator**
A decorator takes a function, wraps it inside a new function called `wrapper`, and returns the wrapper.

In [2]:
def announce(func):
    def wrapper():
        print("--- before ---")
        func()                 # Call the original function
        print("--- after ---")
    return wrapper

def say_hi():
    print("Nomoskar")

say_hi = announce(say_hi)      # Decorating by hand
say_hi()

--- before ---
Nomoskar
--- after ---


## **The @ Symbol**
Writing `@announce` above a function is just a **short form** of `say_bye = announce(say_bye)`.

In [3]:
@announce
def say_bye():
    print("Aschi, bye")

say_bye()

--- before ---
Aschi, bye
--- after ---


## **Decorators for Functions with Inputs**
We use `*args` and `**kwargs` in the wrapper so that it works for **any** function, whatever its inputs are.

In [4]:
def announce(func):
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__} with {args} {kwargs}")
        result = func(*args, **kwargs)
        print(f"{func.__name__} returned {result!r}")
        return result
    return wrapper

@announce
def fare(km, ac=False):
    return round(km * (2.5 if ac else 1.2), 2)

fare(40)
fare(40, ac=True)

Calling fare with (40,) {}
fare returned 48.0
Calling fare with (40,) {'ac': True}
fare returned 100.0


100.0

## **Example from Remote Sensing: A Timer Decorator**
Processing satellite images can take a long time. A **timer decorator** tells us how long each step takes, so we know which step to make faster.

In [5]:
import time

def timer(func):
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        end = time.perf_counter()
        print(f"[{func.__name__}] took {end - start:.4f} seconds")
        return result
    return wrapper

@timer
def ndvi_for_image(red, nir):
    return [(n - r) / (n + r) if (n + r) else 0 for r, n in zip(red, nir)]

@timer
def mean_value(values):
    return sum(values) / len(values)

import random
random.seed(1)
red = [random.uniform(0.02, 0.2) for _ in range(500_000)]
nir = [random.uniform(0.1, 0.5) for _ in range(500_000)]

ndvi = ndvi_for_image(red, nir)
print("Average NDVI:", round(mean_value(ndvi), 3))

[ndvi_for_image] took 0.0351 seconds
[mean_value] took 0.0030 seconds
Average NDVI: 0.442


Let us understand the code above:
- We did not change `ndvi_for_image` or `mean_value` at all. We only added `@timer` on top.
- The wrapper notes the time, runs the real function, notes the time again, and prints the difference.

## **A Decorator That Counts Calls**

In [6]:
def count_calls(func):
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return func(*args, **kwargs)
    wrapper.calls = 0
    return wrapper

@count_calls
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

fib(15)
print("fib was called", fib.calls, "times")

fib was called 1973 times


## **Our Own Memoize Decorator (Remembering Old Answers)**

In [7]:
def memoize(func):
    cache = {}
    def wrapper(n):
        if n not in cache:
            cache[n] = func(n)
        return cache[n]
    return wrapper

@memoize
def fib_fast(n):
    return n if n < 2 else fib_fast(n - 1) + fib_fast(n - 2)

print(fib_fast(80))

23416728348467685


## **A Small Problem: The Name is Lost**
After decorating, the function's name and docstring become those of `wrapper`. We will fix this tomorrow with `functools.wraps`.

In [8]:
@timer
def area(r):
    """Area of a circle"""
    return 3.14159 * r * r

print(area.__name__, area.__doc__)

wrapper None


## **Practice Problem: A Login Check Decorator**

In [9]:
current_user = None

def login_required(func):
    def wrapper(*args, **kwargs):
        if current_user is None:
            print("Please log in first")
            return None
        return func(*args, **kwargs)
    return wrapper

@login_required
def view_balance():
    return f"{current_user}'s balance: Rs 12,450"

print(view_balance())
current_user = "Sujata"
print(view_balance())

Please log in first
None
Sujata's balance: Rs 12,450


In [10]:
# Quick question
# What will be printed?
# def twice(f):
#     def w():
#         f(); f()
#     return w
# @twice
# def hi():
#     print("hi")
# hi()